# QaptaanLM 5-Domain Benchmark Suite: Base vs CPT vs SFT
### Head-to-Head 3-Way Evaluation across 5 Official Benchmark Domains

This benchmark notebook evaluates the official **QaptaanLM-0.75B-Instruct** (`kaptaan45/QaptaanLM-0.75B-Instruct`) on Dual Tesla T4 GPUs across 5 official benchmarks and computes the 3-Way Leaderboard:
- **Coding**: HumanEval (164 tasks, `pass@1`), MBPP (257 tasks, `pass@1`)
- **Math Reasoning**: GSM8K (200 problems, 5-shot CoT Accuracy)
- **General Knowledge**: MMLU (250 questions, 5-shot MCQ Accuracy)
- **Scientific Reasoning**: ARC-Challenge (200 questions, 25-shot MCQ Accuracy)

**Hardware Accelerator**: Select **GPU T4 x2** from the Kaggle GUI right sidebar dropdown.

## 1. Clean Dependencies & Setup Environment

In [ ]:
!pip uninstall -y torchvision torchaudio
!pip install -q --upgrade transformers accelerate datasets safetensors tabulate sympy


## 2. Hardware Verification & Model Loading on Dual Tesla T4 GPUs

In [ ]:
import gc, os, sys, time, json, math, re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import HTML, display
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'[OK] Hardware: {DEVICE} ({NUM_GPUS} GPUs available)')
for i in range(NUM_GPUS):
    print(f'     GPU {i}: {torch.cuda.get_device_name(i)} (Capability: {torch.cuda.get_device_capability(i)})')
print(f'[OK] Precision: {DTYPE}')

MODEL_ID = 'kaptaan45/QaptaanLM-0.75B-Instruct'
print(f'\nLoading {MODEL_ID} directly from Hugging Face Hub into Dual T4 GPU memory...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'✓ Successfully loaded model with {total_params:,} parameters ({total_params/1e6:.2f}M)!')

stop_token_ids = [tokenizer.eos_token_id]
im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
if im_end_id is not None and im_end_id not in stop_token_ids:
    stop_token_ids.append(im_end_id)
print(f'Stop token IDs: {stop_token_ids}')


## 3. Official Datasets & Sandbox Setup

In [ ]:
# Baseline results from official evaluation runs
BASE_RESULTS = {
    'HumanEval': {'score': 19.51, 'passed': 32, 'total': 164, 'category': 'Coding', 'metric': 'pass@1'},
    'MBPP': {'score': 1.17, 'passed': 3, 'total': 257, 'category': 'Coding', 'metric': 'pass@1'},
    'GSM8K': {'score': 42.00, 'passed': 84, 'total': 200, 'category': 'Math Reasoning', 'metric': 'accuracy'},
    'MMLU': {'score': 44.00, 'passed': 110, 'total': 250, 'category': 'General Knowledge', 'metric': 'accuracy'},
    'ARC-Challenge': {'score': 64.50, 'passed': 129, 'total': 200, 'category': 'Reasoning & Science', 'metric': 'accuracy'},
}

CPT_RESULTS = {
    'HumanEval': {'score': 0.61, 'passed': 1, 'total': 164, 'category': 'Coding', 'metric': 'pass@1'},
    'MBPP': {'score': 0.00, 'passed': 0, 'total': 257, 'category': 'Coding', 'metric': 'pass@1'},
    'GSM8K': {'score': 0.50, 'passed': 1, 'total': 200, 'category': 'Math Reasoning', 'metric': 'accuracy'},
    'MMLU': {'score': 4.00, 'passed': 10, 'total': 250, 'category': 'General Knowledge', 'metric': 'accuracy'},
    'ARC-Challenge': {'score': 11.50, 'passed': 23, 'total': 200, 'category': 'Reasoning & Science', 'metric': 'accuracy'},
}

def run_sandbox_test(code: str, test_code: str, entry_point: Optional[str] = None) -> bool:
    full_program = (
        'import sys, math, collections, itertools, functools, re, heapq, bisect\n'
        'from typing import Any, Dict, List, Optional, Set, Tuple, Union, Callable\n\n'
        f'{code}\n\n'
        f'{test_code}\n'
    )
    if entry_point and f'check({entry_point})' not in test_code and 'check(' in test_code:
        full_program += f'\ncheck({entry_point})\n'
    try:
        local_ns = {}
        exec(full_program, {}, local_ns)
        return True
    except Exception:
        return False

def clean_humaneval_code(prompt: str, gen: str) -> str:
    if '```python' in gen:
        return gen.split('```python')[1].split('```')[0].strip()
    elif '```' in gen:
        return gen.split('```')[1].split('```')[0].strip()
    for stop_str in ['\ndef ', '\nclass ', '\nif __name__', '\nprint(', '\nassert ']:
        if stop_str in gen:
            gen = gen.split(stop_str)[0]
    return prompt + gen

def extract_mbpp_code(gen: str) -> str:
    if '```python' in gen:
        return gen.split('```python')[1].split('```')[0].strip()
    elif '```' in gen:
        return gen.split('```')[1].split('```')[0].strip()
    for stop_str in ['\nassert ', '\nif __name__', '\nprint(']:
        if stop_str in gen:
            gen = gen.split(stop_str)[0]
    lines = [l for l in gen.split('\n') if not l.startswith('##') and not l.startswith('Explanation:')]
    return '\n'.join(lines).strip()

def extract_gsm8k_answer(text: str) -> Optional[str]:
    if '####' in text:
        return text.split('####')[-1].replace(',', '').replace('$', '').strip()
    match = re.search(r'[Tt]he answer is:?\s*([+-]?\$?[\d,]+(?:\.\d+)?)', text)
    if match:
        return match.group(1).replace(',', '').replace('$', '').strip()
    numbers = re.findall(r'[-+]?\d*\.?\d+', text.replace(',', ''))
    return numbers[-1].strip() if numbers else None

def is_math_equal(pred: Optional[str], target: str) -> bool:
    if not pred or not target:
        return False
    p, t = pred.strip().replace('$', '').replace(',', ''), target.strip().replace('$', '').replace(',', '')
    if p.lower() == t.lower():
        return True
    try:
        if math.isclose(float(p), float(t), rel_tol=1e-4):
            return True
    except Exception:
        pass
    return False

def extract_mcq_answer(text: str, choices: List[str] = ['A', 'B', 'C', 'D']) -> str:
    pattern = '|'.join(choices)
    m = re.search(rf'[Tt]he (?:correct )?answer is:?\s*\(?([{pattern}])\)?', text)
    if m:
        return m.group(1).upper()
    m = re.findall(rf'\b([{pattern}])\b', text)
    if m:
        return m[-1].upper()
    return text.strip()[:1].upper() if text.strip() else 'A'

from datasets import load_dataset
print('Downloading Official Datasets...')
DATASETS = {}
DATASETS['HumanEval'] = list(load_dataset('openai_humaneval', split='test'))
DATASETS['MBPP'] = list(load_dataset('google-research-datasets/mbpp', 'sanitized', split='test'))
DATASETS['GSM8K'] = list(load_dataset('gsm8k', 'main', split='test'))[:200]
DATASETS['MMLU'] = list(load_dataset('cais/mmlu', 'all', split='test'))[:250]
DATASETS['ARC-Challenge'] = list(load_dataset('ai2_arc', 'ARC-Challenge', split='test'))[:200]
print(f'✓ Loaded: HumanEval ({len(DATASETS["HumanEval"])}), MBPP ({len(DATASETS["MBPP"])}), GSM8K ({len(DATASETS["GSM8K"])}), MMLU ({len(DATASETS["MMLU"])}), ARC ({len(DATASETS["ARC-Challenge"])})')


## 4. Run SFT Benchmark Evaluation on Dual T4 GPUs

In [ ]:
sft_results = {}

# A. HumanEval
tasks = DATASETS['HumanEval']
passed = 0
for row in tqdm(tasks, desc='[QaptaanLM-SFT] HumanEval'):
    prompt_text = f'Complete the following Python function:\n```python\n{row["prompt"]}\n```'
    chat_text = f'<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n```python\n{row["prompt"]}'
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=stop_token_ids, pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.05, use_cache=True)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    full_code = clean_humaneval_code(row['prompt'], gen)
    if run_sandbox_test(full_code, row['test'], row['entry_point']):
        passed += 1
score = round(passed / len(tasks) * 100.0, 2)
sft_results['HumanEval'] = {'score': score, 'passed': passed, 'total': len(tasks), 'category': 'Coding', 'metric': 'pass@1'}
print(f' -> HumanEval pass@1: {score}% ({passed}/{len(tasks)})')

# B. MBPP
tasks = DATASETS['MBPP']
passed = 0
for row in tqdm(tasks, desc='[QaptaanLM-SFT] MBPP'):
    prompt_text = f'Write a Python function to solve this task:\n{row["prompt"]}\nProvide only executable Python code.'
    chat_text = f'<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n```python\n'
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=stop_token_ids, pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.05, use_cache=True)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    code = extract_mbpp_code(gen)
    test_code = (row.get('test_setup_code', '') + '\n') + '\n'.join(row.get('test_list', []))
    if run_sandbox_test(code, test_code):
        passed += 1
score = round(passed / len(tasks) * 100.0, 2)
sft_results['MBPP'] = {'score': score, 'passed': passed, 'total': len(tasks), 'category': 'Coding', 'metric': 'pass@1'}
print(f' -> MBPP pass@1: {score}% ({passed}/{len(tasks)})')

# C. GSM8K
tasks = DATASETS['GSM8K']
correct = 0
for row in tqdm(tasks, desc='[QaptaanLM-SFT] GSM8K'):
    chat_text = f'<|im_start|>user\nSolve this math word problem step by step:\n{row["question"]}<|im_end|>\n<|im_start|>assistant\nLet\'s think step by step.'
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False, eos_token_id=stop_token_ids, pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.05, use_cache=True)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = extract_gsm8k_answer(gen)
    gt = row['answer'].split('####')[-1].strip() if '####' in row['answer'] else row['answer'].strip()
    if is_math_equal(pred, gt):
        correct += 1
score = round(correct / len(tasks) * 100.0, 2)
sft_results['GSM8K'] = {'score': score, 'passed': correct, 'total': len(tasks), 'category': 'Math Reasoning', 'metric': 'accuracy'}
print(f' -> GSM8K Accuracy: {score}% ({correct}/{len(tasks)})')

# D. MMLU
tasks = DATASETS['MMLU']
correct = 0
letters = ['A', 'B', 'C', 'D']
for row in tqdm(tasks, desc='[QaptaanLM-SFT] MMLU'):
    choices_str = '\n'.join([f'({letters[i]}) {c}' for i, c in enumerate(row['choices'])])
    prompt_text = f'Answer the following multiple choice question by giving the correct letter choice:\n{row["question"]}\n{choices_str}'
    chat_text = f'<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\nThe correct answer is: ('
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False, eos_token_id=stop_token_ids, pad_token_id=tokenizer.pad_token_id)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = extract_mcq_answer(gen, letters)
    gt = letters[row['answer']] if isinstance(row['answer'], int) else str(row['answer'])
    if pred == gt:
        correct += 1
score = round(correct / len(tasks) * 100.0, 2)
sft_results['MMLU'] = {'score': score, 'passed': correct, 'total': len(tasks), 'category': 'General Knowledge', 'metric': 'accuracy'}
print(f' -> MMLU Accuracy: {score}% ({correct}/{len(tasks)})')

# E. ARC-Challenge
tasks = DATASETS['ARC-Challenge']
correct = 0
letters = ['A', 'B', 'C', 'D', 'E']
for row in tqdm(tasks, desc='[QaptaanLM-SFT] ARC-Challenge'):
    choices_str = '\n'.join([f'({row["choices"]["label"][i]}) {c}' for i, c in enumerate(row['choices']['text'])])
    prompt_text = f'Answer this scientific reasoning question by selecting the correct option letter:\n{row["question"]}\n{choices_str}'
    chat_text = f'<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\nThe correct answer is: ('
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False, eos_token_id=stop_token_ids, pad_token_id=tokenizer.pad_token_id)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = extract_mcq_answer(gen, letters)
    gt = row['answerKey'].strip().upper()
    if pred == gt:
        correct += 1
score = round(correct / len(tasks) * 100.0, 2)
sft_results['ARC-Challenge'] = {'score': score, 'passed': correct, 'total': len(tasks), 'category': 'Reasoning & Science', 'metric': 'accuracy'}
print(f' -> ARC-Challenge: {score}% ({correct}/{len(tasks)})')

del model
del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. Render 3-Way Comprehensive Leaderboard & Export Artifacts

In [ ]:
all_benchmarks = ['HumanEval', 'MBPP', 'GSM8K', 'MMLU', 'ARC-Challenge']

html_scorecard = f'''
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1050px; margin: 0 auto;">
  <div style="background: linear-gradient(135deg, #1e293b, #0f172a); padding: 24px; border-radius: 12px; border: 1px solid #334155; margin-bottom: 24px; text-align: center;">
    <h2 style="color: #38bdf8; margin: 0 0 8px 0; font-size: 24px;">QaptaanLM 3-Way Comprehensive Benchmark Leaderboard</h2>
    <p style="color: #94a3b8; margin: 0; font-size: 14px;">
      Base (<strong>Qwen3.5-0.8B-Base</strong>) vs CPT (<strong>QaptaanLM-0.75B</strong>) vs SFT (<strong>QaptaanLM-0.75B-Instruct</strong>)
    </p>
  </div>
  <table style="width: 100%; border-collapse: collapse; background: #0f172a; border-radius: 10px; overflow: hidden; border: 1px solid #334155; font-size: 14px;">
    <thead>
      <tr style="background: #1e293b; color: #94a3b8; text-transform: uppercase; font-size: 12px;">
        <th style="padding: 14px 16px; text-align: left;">Benchmark</th>
        <th style="padding: 14px 16px; text-align: left;">Domain</th>
        <th style="padding: 14px 16px; text-align: center;">Metric</th>
        <th style="padding: 14px 16px; text-align: center;">Base (0.8B)</th>
        <th style="padding: 14px 16px; text-align: center;">CPT (0.75B)</th>
        <th style="padding: 14px 16px; text-align: center; color: #c084fc;">SFT (Instruct)</th>
        <th style="padding: 14px 16px; text-align: center;">SFT vs Base</th>
      </tr>
    </thead>
    <tbody>
'''

md_table = '# QaptaanLM 3-Way Benchmark Leaderboard\n\n'
md_table += '| Benchmark | Domain | Metric | Qwen3.5-0.8B (Base) | QaptaanLM-0.75B (CPT) | **QaptaanLM-0.75B (SFT)** | Delta (SFT vs Base) |\n'
md_table += '| :--- | :--- | :---: | :---: | :---: | :---: | :---: |\n'

for b in all_benchmarks:
    b_data = BASE_RESULTS.get(b, {})
    c_data = CPT_RESULTS.get(b, {})
    s_data = sft_results.get(b, {})
    
    s_base = b_data.get('score', 0.0)
    s_cpt = c_data.get('score', 0.0)
    s_sft = s_data.get('score', 0.0)
    cat = s_data.get('category', 'General')
    metric = s_data.get('metric', 'accuracy')
    
    diff = s_sft - s_base
    diff_color = '#4ade80' if diff > 0 else ('#f87171' if diff < 0 else '#94a3b8')
    diff_str = f'+{diff:.2f}%' if diff > 0 else f'{diff:.2f}%'
    
    html_scorecard += f'''
      <tr style="border-bottom: 1px solid #1e293b;">
        <td style="padding: 12px 16px; color: #f8fafc; font-weight: bold;">{b}</td>
        <td style="padding: 12px 16px; color: #94a3b8;">{cat}</td>
        <td style="padding: 12px 16px; text-align: center; color: #94a3b8;">{metric}</td>
        <td style="padding: 12px 16px; text-align: center; color: #60a5fa;">{s_base:.2f}%</td>
        <td style="padding: 12px 16px; text-align: center; color: #38bdf8;">{s_cpt:.2f}%</td>
        <td style="padding: 12px 16px; text-align: center; color: #c084fc; font-weight: bold;">{s_sft:.2f}%</td>
        <td style="padding: 12px 16px; text-align: center; color: {diff_color}; font-weight: bold;">{diff_str}</td>
      </tr>
    '''
    md_table += f'| **{b}** | {cat} | `{metric}` | {s_base:.2f}% | {s_cpt:.2f}% | **{s_sft:.2f}%** | `{diff_str}` |\n'

html_scorecard += '''
    </tbody>
  </table>
</div>
'''

display(HTML(html_scorecard))

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./benchmark_results')
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / 'benchmark_results.json', 'w', encoding='utf-8') as f:
    json.dump({'base': BASE_RESULTS, 'cpt': CPT_RESULTS, 'sft': sft_results}, f, indent=2)

with open(out_dir / 'benchmark_report.md', 'w', encoding='utf-8') as f:
    f.write(md_table)

with open(out_dir / 'benchmark_leaderboard.html', 'w', encoding='utf-8') as f:
    f.write(html_scorecard)

print(f'\n[OK] Successfully exported artifacts to: {out_dir}')
print(' - benchmark_results.json')
print(' - benchmark_report.md')
print(' - benchmark_leaderboard.html')
